In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

from rasterio.features import rasterize

from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    cohen_kappa_score
)

In [29]:
krishna_hsi = r"C:\Users\snvra\NISAR_Data_project\Data\krishsatemb2019_24\Krishsatemb2024.tif"

with rasterio.open(krishna_hsi) as src:
    hsi = src.read()
    transform = src.transform
    crs = src.crs
    profile = src.profile

# (bands, rows, cols) → (rows, cols, bands)
hsi = np.moveaxis(hsi, 0, -1)

print(hsi.shape)

(2236, 2102, 64)


In [30]:
train = gpd.read_file(r"C:\Users\snvra\NISAR_Data_project\Data\krishna_gt\train_2024.shp")

# Ensure same CRS
if train.crs != crs:
    train = train.to_crs(crs)

print(train.columns)

Index(['OBJECTID', 'label', 'value', 'Shape_Leng', 'Shape_Area', 'geometry'], dtype='object')


In [31]:
label_raster = rasterize(
    [(geom, value) for geom, value in zip(train.geometry, train["value"])],
    out_shape=(hsi.shape[0], hsi.shape[1]),
    transform=transform,
    fill=0,
    dtype=np.uint8
)

print(np.unique(label_raster))

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]


In [32]:
mask = label_raster > 0

X_source = hsi[mask]
y_source = label_raster[mask]

print("Training Samples :", X_source.shape)
print("Labels           :", y_source.shape)

Training Samples : (16698, 64)
Labels           : (16698,)


In [33]:
class_names = (
    train[["value", "label"]]
    .drop_duplicates()
    .sort_values("value")
)

counts = pd.Series(y_source).value_counts().sort_index()

distribution = pd.DataFrame({
    "Class ID": counts.index,
    "Pixels": counts.values
})

distribution = distribution.merge(
    class_names,
    left_on="Class ID",
    right_on="value"
)

distribution = distribution[
    ["Class ID","label","Pixels"]
]

print(distribution)

    Class ID                label  Pixels
0          1       A_marina_dense    2115
1          2    A_marina_moderate     856
2          3  A_marina_sparse_rej    2003
3          4          E_aggalocha     110
4          5         B_gymnorhiza      85
5          6        Rhizophora_sp     545
6          7             RH_BG_AM     443
7          8                MIXED      81
8          9      Sparse_mangrove     879
9         10             Prosopis    2098
10        11         SB_Suaeda_sp    1277
11        12      Creek_Waterbody    1559
12        13              Mudflat    2643
13        14            Sand_bars    2004


In [36]:
# Krishna IDs
keep_classes = [1,2,3,11]

mask = np.isin(y_source, keep_classes)

X_train = X_source[mask]
y_train = y_source[mask]

print(X_train.shape)
print(np.unique(y_train))

(6251, 64)
[ 1  2  3 11]


In [37]:
krishna_to_pichavaram = {
    1: 2,    # A. marina dense
    2: 3,    # A. marina moderate
    3: 4,    # A. marina sparse
    11: 1    # Rhizophora
}

y_train = np.array([krishna_to_pichavaram[v] for v in y_train])

print(np.unique(y_train))

[1 2 3 4]
